# Leak-free Model B on the grouped split (issue #37)

Fine-tunes NT and trains Model B's XGBoost heads on the sequence-grouped
split in `grouped_split/` (#34/#35): 1,445 train sequences (4,118 rows), 180
val (513 rows), 180 test (514 rows), with no sequence shared between splits.
Production Model B's backbone was fine-tuned on the stale
`train.csv`/`val.csv`, which contain almost all Testset sequences.

- **Fine-tune:** `nt_pretrain.py`'s recipe unchanged (`grouped_model_b.FINETUNE_ARGS`),
  on grouped train rows, early-stopped on grouped val rows.
- **XGBoost heads** (`ablation.XGB_HYPERPARAMS`, early-stopped on grouped val),
  on the `Model_B_Predictor` features (8,960-d flattened last hidden state + 4
  physical features):
  - **all-rows:** one sample per train row (as in production);
  - **seq-mean:** one sample per train sequence, labels averaged across replicates.

**One-time setup:** upload to a Google Drive folder, keeping the layout:
`SaCas9_v4.xlsx`, `hybrid_xgb_model.json` (only to construct
`Model_B_Predictor`, whose constructor needs an XGBoost file; its head is not
used here), and `grouped_split/assignments.csv`,
`grouped_split/test_metadata.csv`, `grouped_split/test_checksum.json`. Set
`DRIVE_DIR` below to that folder.

**Outputs:**
- `NT_sacas9_grouped_model/` on Drive (~2 GB): the best fine-tuned checkpoint.
  Download it to an untracked local `xai_test/NT_sacas9_grouped_model/`; it
  loads with `Model_B_Predictor` like `NT_sacas9_fintuned_model/`.
- `grouped_model_b/` (also on Drive, and downloaded as a zip):
  `xgb_all_rows.json`, `xgb_seq_mean.json`, `test_predictions.csv`,
  `training_log.json`. Copy it into your local `xai_test/` and commit it.
  Embeddings are not saved.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded assets

Edit `DRIVE_DIR` to the Drive folder holding the files listed above.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = [
    "SaCas9_v4.xlsx",
    "hybrid_xgb_model.json",
    "grouped_split/assignments.csv",
    "grouped_split/test_metadata.csv",
    "grouped_split/test_checksum.json",
]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. "
        "Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install ViennaRNA CLI (RNAfold / RNAduplex) and Python deps

`Model_B_Predictor.compute_physical_features` needs both binaries on `PATH`.
`nt_pretrain.py` upgraded `transformers`/`accelerate` before fine-tuning; so does this.

In [ ]:
!apt-get -qq update && apt-get -qq install -y vienna-rna
!which RNAfold && which RNAduplex

In [ ]:
!pip install -q --upgrade transformers accelerate
!pip install -q biopython xgboost openpyxl

## 4. Write the repo modules

Embedded from the committed `xai_test/` repo at notebook-build time
(`build_colab_grouped_model_b.py` checks them byte-identical), not fetched at runtime.

In [ ]:
%%writefile test_dataset_verifier.py
import hashlib
import os
import sys

import pandas as pd
import torch


def generate_sha256(sequence: str) -> str:
    return hashlib.sha256(sequence.encode('utf-8')).hexdigest()

def load_dataset(file_path: str) -> pd.DataFrame:
    if file_path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(file_path)
    return pd.read_csv(file_path)

def standardize_columns(full_df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names (`sequence`/`true_score`) and preserve `original_id`."""
    rename_mapping = {}
    if 'sequence' not in full_df.columns and 'input_sequence' in full_df.columns:
        rename_mapping['input_sequence'] = 'sequence'
    if 'true_score' not in full_df.columns and 'score' in full_df.columns:
        rename_mapping['score'] = 'true_score'

    if rename_mapping:
        full_df = full_df.rename(columns=rename_mapping)

    if 'original_id' not in full_df.columns:
        full_df['original_id'] = full_df.index

    return full_df

def compute_split_indices(dataset_size: int, seed: int = 42) -> tuple[list[int], list[int]]:
    """The seed=42 `torch.randperm` split isolating the Testset from the dev pool.

    Returns `(train_val_indices, test_indices)`: `test_indices` (the last 10%
    of the permutation) is exactly what `test_metadata.csv` is derived from;
    `train_val_indices` (the first 90%) is the disjoint dev pool other issues
    (e.g. #13's ablation retraining) build train/val splits from - reusing
    this function guarantees they stay leak-free against the Testset.
    """
    test_size = max(1, int(dataset_size * 0.1))
    train_val_size = dataset_size - test_size

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(dataset_size, generator=generator).tolist()
    return indices[:train_val_size], indices[train_val_size:]

def isolate_and_verify_testset(full_dataset_path: str, output_csv_path: str = 'test_metadata.csv', seed: int = 42):
    print(f"Loading full dataset from {full_dataset_path}...")
    full_df = load_dataset(full_dataset_path)
    dataset_size = len(full_df)

    # 1. 컬럼 매핑 표준화 및 원본 행 인덱스 보존
    full_df = standardize_columns(full_df)

    # 2. PyTorch random_split 재현 (int(dataset_size * 0.1) = 514개)
    _, test_indices = compute_split_indices(dataset_size, seed)
    test_size = len(test_indices)

    # 3. 테스트셋 추출 및 크기 검증
    test_df = full_df.iloc[test_indices].copy().reset_index(drop=True)
    if len(test_df) != test_size:
        raise ValueError(f"Split error: Expected {test_size} test sequences, but got {len(test_df)}.")

    # 4. sample_id (0 ~ test_size-1) 부여 및 서열 해시 생성
    test_df['sample_id'] = range(len(test_df))
    test_df['sequence_hash'] = test_df['sequence'].apply(generate_sha256)

    # 5. 전체 데이터셋 단위 SHA256 체크섬
    dataset_hash = hashlib.sha256(pd.util.hash_pandas_object(test_df['sequence']).values).hexdigest()
    print(f"Testset isolation completed: {len(test_df)} samples")
    print(f"Dataset SHA256 Checksum: {dataset_hash}")

    # 6. 필수 메타데이터 칼럼 저장
    required_cols = ['sample_id', 'original_id', 'sequence', 'true_score', 'sequence_hash']
    for col in required_cols:
        if col not in test_df.columns:
            raise KeyError(f"Missing required column in dataset: {col}")

    meta_df = test_df[required_cols]
    meta_df.to_csv(output_csv_path, index=False)
    print(f"Saved isolated test metadata to {output_csv_path}")

    return meta_df, dataset_hash

if __name__ == "__main__":
    script_dir = os.path.dirname(os.path.abspath(__file__))

    if len(sys.argv) > 1:
        target_file = sys.argv[1]
        if not os.path.isabs(target_file):
            target_file = os.path.join(script_dir, target_file)
    else:
        target_file = os.path.join(script_dir, "SaCas9_v4.xlsx")

    try:
        isolate_and_verify_testset(target_file)
    except FileNotFoundError:
        print(f"File not found: {target_file}")
        print(f"현재 스크립트 위치({script_dir})에 데이터 파일이 존재하는지 확인해 주세요.")

In [ ]:
%%writefile grouped_split.py
"""Sequence-grouped train/val/test split with cell_line_block (issue #34).

The rows of `SaCas9_v4.xlsx` are cell-line replicates: its 5,145 rows hold
only 1,805 distinct sequences, so the row-level seed=42 split in
`test_dataset_verifier.py` puts replicates of the same sequence on both sides
of the Testset boundary. This module splits by **sequence group** instead:

- Rows are grouped by normalized sequence (strip + upper-case). With a fixed
  seed, ~10% of the sequences go to the Testset and the same number to val;
  the rest are train. Every replicate of a sequence lands in the same split.
- Each row gets a `cell_line_block` (L1/L2/L3) from its row position
  (rows 0-1534 / 1535-3339 / 3340-5144). It is metadata only - not a model
  input and not mapped to a cell-line name, since that mapping is unconfirmed.

Read-only with respect to every existing file; writes only into
`grouped_split/` (`assignments.csv`, `summary.json`).

Usage: python grouped_split.py [--seed 42]
"""

import argparse
import json
from collections.abc import Iterable, Sequence
from pathlib import Path

import numpy as np
import pandas as pd
from test_dataset_verifier import load_dataset, standardize_columns

BASE_DIR = Path(__file__).resolve().parent
DATASET_PATH = BASE_DIR / "dataset" / "SaCas9_v4.xlsx"
OUTPUT_DIR = BASE_DIR / "grouped_split"

SPLITS = ("train", "val", "test")
CELL_LINE_BLOCK_SIZES = (1535, 1805, 1805)
TEST_FRAC = 0.1
VAL_FRAC = 0.1
OUTPUT_COLUMNS = ["original_id", "normalized_sequence", "cell_line_block", "split"]


def load_full_dataset(path: Path = DATASET_PATH) -> pd.DataFrame:
    return standardize_columns(load_dataset(str(path)))


def block_labels(sizes: Sequence[int] = CELL_LINE_BLOCK_SIZES) -> list[str]:
    return [f"L{i + 1}" for i in range(len(sizes))]


def normalize_sequences(sequences: pd.Series) -> pd.Series:
    return sequences.astype(str).str.strip().str.upper()


def split_sequence_groups(
    sequences: Iterable[str],
    seed: int = 42,
    test_frac: float = TEST_FRAC,
    val_frac: float = VAL_FRAC,
) -> dict[str, str]:
    """Map each distinct sequence to `train`/`val`/`test`.

    Sequences are sorted before the seeded permutation, so the result depends
    only on the set of sequences and the seed, not on row order.
    """
    unique = np.array(sorted(set(sequences)))
    n = len(unique)
    n_test = int(n * test_frac)
    n_val = int(n * val_frac)

    shuffled = unique[np.random.default_rng(seed).permutation(n)]
    return {
        **{seq: "test" for seq in shuffled[:n_test]},
        **{seq: "val" for seq in shuffled[n_test : n_test + n_val]},
        **{seq: "train" for seq in shuffled[n_test + n_val :]},
    }


def assign_cell_line_blocks(
    n_rows: int, sizes: Sequence[int] = CELL_LINE_BLOCK_SIZES
) -> pd.Series:
    """`L1`/`L2`/... per row position, with block lengths given by `sizes`."""
    if sum(sizes) != n_rows:
        raise ValueError(
            f"cell_line_block sizes {tuple(sizes)} sum to {sum(sizes)}, "
            f"but the dataset has {n_rows} rows"
        )
    return pd.Series(np.repeat(block_labels(sizes), sizes), name="cell_line_block")


def validate_cell_line_blocks(
    df: pd.DataFrame, sizes: Sequence[int] = CELL_LINE_BLOCK_SIZES
) -> None:
    """Block sizes must match `sizes`, and no sequence may repeat within a block."""
    counts = df["cell_line_block"].value_counts()
    actual = tuple(int(counts.get(label, 0)) for label in block_labels(sizes))
    if actual != tuple(sizes):
        raise ValueError(f"cell_line_block sizes {actual}, expected {tuple(sizes)}")

    per_block = df.groupby("cell_line_block")["normalized_sequence"]
    repeated = per_block.size() - per_block.nunique()
    if (repeated > 0).any():
        raise ValueError(
            f"Repeated sequences within cell_line_block: "
            f"{repeated[repeated > 0].to_dict()}"
        )


def build_grouped_split(
    full_df: pd.DataFrame,
    seed: int = 42,
    block_sizes: Sequence[int] = CELL_LINE_BLOCK_SIZES,
) -> pd.DataFrame:
    """Per-row assignments: `original_id`, `normalized_sequence`, `cell_line_block`, `split`."""
    full_df = full_df.reset_index(drop=True)
    assignments = pd.DataFrame(
        {
            "original_id": full_df["original_id"],
            "normalized_sequence": normalize_sequences(full_df["sequence"]),
            "cell_line_block": assign_cell_line_blocks(len(full_df), block_sizes),
        }
    )
    validate_cell_line_blocks(assignments, block_sizes)

    split_of = split_sequence_groups(assignments["normalized_sequence"], seed=seed)
    assignments["split"] = assignments["normalized_sequence"].map(split_of)
    return assignments[OUTPUT_COLUMNS]


def summarize(assignments: pd.DataFrame) -> dict:
    """Row and sequence counts per split, with rows broken down by `cell_line_block`."""
    blocks = sorted(set(block_labels()) | set(assignments["cell_line_block"]))

    def counts(part: pd.DataFrame) -> dict:
        by_block = part["cell_line_block"].value_counts()
        return {
            "rows": len(part),
            "sequences": int(part["normalized_sequence"].nunique()),
            "rows_by_cell_line_block": {b: int(by_block.get(b, 0)) for b in blocks},
        }

    summary = {s: counts(assignments[assignments["split"] == s]) for s in SPLITS}
    summary["total"] = counts(assignments)
    return summary


def main(argv: list[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(argv)

    assignments = build_grouped_split(load_full_dataset(), seed=args.seed)
    summary = summarize(assignments)

    OUTPUT_DIR.mkdir(exist_ok=True)
    assignments.to_csv(OUTPUT_DIR / "assignments.csv", index=False)
    with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
        json.dump({"seed": args.seed, "counts": summary}, f, indent=2)

    for split in (*SPLITS, "total"):
        split_counts = summary[split]
        print(
            f"{split:>5}: {split_counts['rows']:>5} rows, "
            f"{split_counts['sequences']:>5} sequences, "
            f"rows by cell_line_block {split_counts['rows_by_cell_line_block']}"
        )
    print(f"Wrote {OUTPUT_DIR / 'assignments.csv'} and {OUTPUT_DIR / 'summary.json'}")
    return summary


if __name__ == "__main__":
    main()


In [ ]:
%%writefile grouped_testset.py
"""Export the grouped Testset as test_metadata.csv-style metadata (issue #35).

Takes the `test` rows of the sequence-grouped split (#34) - every replicate
row of each test sequence, in source row order - and writes them with the
`test_metadata.csv` columns plus `cell_line_block`:

- `sample_id` is a fresh 0..N-1 Testset-internal index in row order, distinct
  from `original_id` (the row's index in `SaCas9_v4.xlsx`).
- `sequence_hash` is the per-row SHA-256 of `sequence`, as in
  `test_dataset_verifier.py`; the whole-column checksum is built the same way
  that verifier builds it, and is recorded in `test_checksum.json` together
  with the SHA-256 of the written CSV bytes.

Leaves the existing `test_metadata.csv` untouched; writes only into
`grouped_split/` (`test_metadata.csv`, `test_checksum.json`).

Usage: python grouped_testset.py [--seed 42] [--output-dir grouped_split]
"""

import argparse
import hashlib
import json
from pathlib import Path

import pandas as pd
from grouped_split import (
    OUTPUT_DIR,
    build_grouped_split,
    load_full_dataset,
    normalize_sequences,
)
from test_dataset_verifier import generate_sha256

TESTSET_COLUMNS = [
    "sample_id",
    "original_id",
    "sequence",
    "true_score",
    "sequence_hash",
    "cell_line_block",
]
TESTSET_FILENAME = "test_metadata.csv"
CHECKSUM_FILENAME = "test_checksum.json"


def column_checksum(sequences: pd.Series) -> str:
    """Whole-column SHA-256, built as in `isolate_and_verify_testset`.

    `hash_pandas_object` includes the index, so pass the column with its
    0..N-1 index (as read from or written to the CSV).
    """
    return hashlib.sha256(pd.util.hash_pandas_object(sequences).values).hexdigest()


def build_grouped_testset(
    full_df: pd.DataFrame, assignments: pd.DataFrame
) -> pd.DataFrame:
    """The `test` rows of `full_df` per `assignments`, in `TESTSET_COLUMNS` form."""
    full_df = full_df.reset_index(drop=True)
    assignments = assignments.reset_index(drop=True)
    if not full_df["original_id"].equals(assignments["original_id"]):
        raise ValueError("assignments do not line up row-for-row with the dataset")

    is_test = (assignments["split"] == "test").to_numpy()
    testset = full_df.loc[is_test, ["original_id", "sequence", "true_score"]]
    testset["cell_line_block"] = assignments.loc[is_test, "cell_line_block"]
    testset = testset.reset_index(drop=True)

    testset["sample_id"] = range(len(testset))
    testset["sequence_hash"] = testset["sequence"].apply(generate_sha256)
    return testset[TESTSET_COLUMNS]


def main(argv: list[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output-dir", type=Path, default=OUTPUT_DIR)
    args = parser.parse_args(argv)

    full_df = load_full_dataset()
    assignments = build_grouped_split(full_df, seed=args.seed)
    testset = build_grouped_testset(full_df, assignments)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = args.output_dir / TESTSET_FILENAME
    # Fixed line ending so `file_sha256` matches across Windows and Colab.
    testset.to_csv(csv_path, index=False, lineterminator="\n")

    record = {
        "seed": args.seed,
        "rows": len(testset),
        "sequences": int(normalize_sequences(testset["sequence"]).nunique()),
        "dataset_sha256": column_checksum(testset["sequence"]),
        "file_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    }
    with open(args.output_dir / CHECKSUM_FILENAME, "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2)

    print(f"Grouped Testset: {record['rows']} rows, {record['sequences']} sequences")
    print(f"Dataset SHA256 Checksum: {record['dataset_sha256']}")
    print(f"Wrote {csv_path} and {args.output_dir / CHECKSUM_FILENAME}")
    return record


if __name__ == "__main__":
    main()


In [ ]:
%%writefile model_b_wrapper.py
import os
import pickle
import re
import subprocess

import joblib
import numpy as np
import torch
import xgboost as xgb
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import gc_fraction
from transformers import AutoModelForMaskedLM, AutoTokenizer


class Model_B_Predictor:
    def __init__(self, nt_model_dir: str, xgb_model_path: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading NT backbone from {nt_model_dir}...")
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        self.nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_dir).to(
            self.device
        )
        self.nt_model.eval()

        print(f"Loading XGBoost model from {xgb_model_path}...")
        self.xgb_model = self._load_xgb_model(xgb_model_path)

    def _load_xgb_model(self, path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"XGBoost model file not found: {path}")

        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
            try:
                return joblib.load(path)
            except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
                model = xgb.XGBRegressor()
                model.load_model(path)
                return model

    def extract_nt_embeddings(self, sequences: list[str]) -> np.ndarray:
        inputs = self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.nt_model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]

            batch_size, seq_len, hidden_dim = last_hidden_state.shape
            assert seq_len == 7, f"Expected 7 tokens, got {seq_len}"
            assert hidden_dim == 1280, f"Expected 1280 hidden dim, got {hidden_dim}"

            flattened_embeddings = last_hidden_state.view(batch_size, -1).cpu().numpy()
        return flattened_embeddings

    def _get_guide_rna(self, dna_seq: str) -> str:
        complement_map = str.maketrans("ATCGatcg", "UAGCuagc")
        return dna_seq.translate(complement_map)[::-1]

    # RNAfold/RNAduplex are external processes: each subprocess.run() call pays
    # OS process-spawn overhead regardless of how little work it does. Both CLIs
    # accept many records over a single stdin stream, so batching sequences into
    # one call (chunked, to keep any single call's stdin/stdout bounded) turns an
    # O(n) process-spawn cost into O(n / batch_size) - the dominant cost at
    # ISM-sweep scale (tens of thousands of mutants). Deliberate scope override of
    # issue #9's "RNAfold/RNAduplex call batching" Out-of-Scope note, requested
    # directly by the user; verified to reproduce the exact same per-sequence
    # energy values as the prior one-process-per-sequence implementation.
    PHYSICAL_FEATURE_BATCH_SIZE = 500

    @staticmethod
    def _chunked(items: list, size: int):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def _calc_mfe_batch(self, seqs: list[str]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(seqs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAfold CLI 호출 (--noPS: 이미지 생성 방지)
            proc = subprocess.run(
                ["RNAfold", "--noPS"],
                input="".join(f"{seq}\n" for seq in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # Each record prints "<sequence>\n<dot-bracket structure> (<energy>)\n".
            # The structure diagram's own parentheses never have a digit directly
            # inside them, so this regex matches exactly one energy per record,
            # in input order. 출력 예시: ... ( -4.20)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAfold batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _calc_duplex_dg_batch(self, pairs: list[tuple[str, str]]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(pairs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAduplex CLI 호출
            proc = subprocess.run(
                ["RNAduplex"],
                input="".join(f"{guide}\n{target}\n" for guide, target in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # 출력 예시: ... :  1,36 :   1,36 (-32.50)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAduplex batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def compute_physical_features(self, sequences: list[str]) -> np.ndarray:
        # 1. MFE (RNAfold.exe), batched across all sequences
        mfe_values = self._calc_mfe_batch(sequences)

        # 2. Duplex ΔG (RNAduplex.exe), batched across all sequences
        guide_rnas = [self._get_guide_rna(seq) for seq in sequences]
        dg_values = self._calc_duplex_dg_batch(list(zip(guide_rnas, sequences)))

        features = []
        for seq, mfe, dg in zip(sequences, mfe_values, dg_values):
            # 3. Tm (must match training-time nn_table=DNA_NN4 in nt_feature_stacking.py)
            tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

            # 4. GC Content
            gc = gc_fraction(seq) * 100.0

            features.append([mfe, dg, tm, gc])
        return np.array(features, dtype=np.float32)

    def predict(self, sequences: list[str]) -> np.ndarray:
        embeddings = self.extract_nt_embeddings(sequences)
        physical_feats = self.compute_physical_features(sequences)

        final_features = np.hstack([embeddings, physical_feats])
        assert final_features.shape[1] == 8964, (
            f"Dimension mismatch: Expected 8964, got {final_features.shape[1]}"
        )

        preds = self.xgb_model.predict(final_features)
        return np.array(preds).flatten()


In [ ]:
%%writefile ism_sweep.py
"""In-silico saturation mutagenesis (ISM) sweep (Step 3, see CONTEXT.md / plan_realize.md).

Generates every position x alternative-base substitution for a set of 36bp
Testset sequences, scores wild-type and mutants through a predictor's own
`predict()`, and reduces the result to a relative delta-score sensitivity
array. Works against any predictor exposing `predict(sequences) -> array`
(duck-typed to `Model_A_Predictor` / `Model_B_Predictor`), so Model B's
per-mutant physical-feature recomputation (MFE, dG, Tm, GC) happens for free:
every mutant is dispatched as a full sequence, never a cached wild-type value
or a bare delta.
"""

from collections.abc import Sequence

import numpy as np

BASES = ("A", "C", "G", "T")

# Relative delta-Score is undefined (unbounded) as Score_WT -> 0; scores at or
# below this magnitude are masked to NaN instead of producing a spurious ratio.
RELATIVE_DELTA_NAN_THRESHOLD = 1e-4

# Neither Model_A_Predictor nor Model_B_Predictor batches internally - predict()
# runs whatever list it's given as one forward pass. An ISM sweep can hand it
# tens of thousands of mutants at once, which is large enough to exhaust GPU
# memory (observed: CUDA OOM inside Model A's LSTM forward pass on a 14.56GB
# T4 trying to allocate 20.71 GiB for one such call). Chunking dispatch here
# keeps peak memory bounded regardless of sweep size, without changing either
# predictor's own predict() contract.
DEFAULT_PREDICT_BATCH_SIZE = 256


def alt_bases_for(wt_base: str) -> list[str]:
    """The 3 non-wild-type bases, in fixed lexicographic order."""
    wt_base = wt_base.upper()
    if wt_base not in BASES:
        raise ValueError(f"Not a valid base: {wt_base!r}")
    return sorted(set(BASES) - {wt_base})


def generate_ism_mutants(sequence: str) -> list[str]:
    """All position x alternative-base substitutions for `sequence`.

    Ordered position-major (0..len-1), alternative-base lexicographic-minor,
    matching the `ism_delta` array's (position, alternative_base) axes.
    """
    sequence = sequence.upper()
    mutants = []
    for pos, wt_base in enumerate(sequence):
        for alt_base in alt_bases_for(wt_base):
            mutants.append(sequence[:pos] + alt_base + sequence[pos + 1 :])
    return mutants


def dispatch_in_batches(
    fn, items: list[str], batch_size: int, stack=np.concatenate
) -> np.ndarray:
    """Call `fn()` in bounded chunks and combine the results with `stack`.

    Generic version of the chunking `predict_in_batches` needs for 1D score
    arrays; `step4_ablation.py` reuses this directly (with `stack=np.vstack`)
    to batch `Model_B_Predictor.extract_nt_embeddings()`'s 2D output the same
    way, instead of re-deriving the chunk loop.
    """
    if not items:
        return np.array([])
    chunks = [fn(items[i : i + batch_size]) for i in range(0, len(items), batch_size)]
    return stack(chunks)


def predict_in_batches(predictor, sequences: list[str], batch_size: int) -> np.ndarray:
    """Dispatch `predictor.predict()` in bounded chunks (shared with mismatch_profiling.py)."""
    if not sequences:
        return np.array([], dtype=np.float64)

    def _predict_chunk(chunk):
        return np.asarray(predictor.predict(chunk), dtype=np.float64)

    return dispatch_in_batches(
        _predict_chunk, sequences, batch_size, stack=np.concatenate
    )


def relative_delta(
    wt_scores: np.ndarray, mutant_scores: np.ndarray, nan_threshold: float
) -> np.ndarray:
    """`(mutant - wt) / |wt|` per sample, NaN-masked wherever `|wt| <= nan_threshold`.

    `wt_scores` is 1D (one value per sample); `mutant_scores` broadcasts against
    it along its leading axis - shape `(n, positions, alt_bases)` for the ISM
    sweep, `(n, scenarios)` for complex mismatch profiling. Shared so both
    modules apply the exact same guardrail (see `RELATIVE_DELTA_NAN_THRESHOLD`).
    """
    broadcast_shape = (wt_scores.shape[0],) + (1,) * (mutant_scores.ndim - 1)
    wt_broadcast = wt_scores.reshape(broadcast_shape)
    with np.errstate(divide="ignore", invalid="ignore"):
        delta = (mutant_scores - wt_broadcast) / np.abs(wt_broadcast)
    delta[np.abs(wt_scores) <= nan_threshold] = np.nan
    return delta


def compute_ism_delta(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> np.ndarray:
    """Run the full ISM sweep for one predictor.

    Returns `ism_delta`, shape `(len(sequences), seq_len, 3)`: the relative
    score delta `(Score_mutant - Score_WT) / |Score_WT|` per position x
    lexicographic alternative base, NaN-masked wherever `|Score_WT| <=
    nan_threshold`. `predict_batch_size` bounds how many sequences reach
    `predictor.predict()` per call; lower it if you still hit an
    out-of-memory error.
    """
    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError("All sequences must share the same length for the ISM sweep.")

    wt_scores = predict_in_batches(predictor, sequences, predict_batch_size)

    mutants: list[str] = []
    for sequence in sequences:
        mutants.extend(generate_ism_mutants(sequence))

    mutant_scores = predict_in_batches(predictor, mutants, predict_batch_size)
    mutant_scores = mutant_scores.reshape(len(sequences), seq_len, 3)

    return relative_delta(wt_scores, mutant_scores, nan_threshold)


def run_ism_sweep(
    sequences: Sequence[str],
    predictor_a,
    predictor_b,
    predict_batch_size: int = DEFAULT_PREDICT_BATCH_SIZE,
) -> dict[str, np.ndarray]:
    """Run the ISM sweep for both models over the same Testset sequences."""
    return {
        "ism_delta_model_a": compute_ism_delta(
            sequences, predictor_a, predict_batch_size=predict_batch_size
        ),
        "ism_delta_model_b": compute_ism_delta(
            sequences, predictor_b, predict_batch_size=predict_batch_size
        ),
    }


In [ ]:
%%writefile paired_bootstrap.py
"""Paired bootstrap significance testing for Model B ablation (Step 4-1, issue #13).

Each bootstrap draw resamples the SAME index set (with replacement) for
y_true and every compared variant's predictions together, so the comparison
stays paired - it isolates the metric delta itself rather than treating the
two variants as independent samples with their own sampling noise.
"""

from collections.abc import Callable

import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

MetricFn = Callable[[np.ndarray, np.ndarray], float]

METRIC_FUNCS: dict[str, MetricFn] = {
    "spearman": lambda y_true, y_pred: spearmanr(y_true, y_pred)[0],
    "pearson": lambda y_true, y_pred: pearsonr(y_true, y_pred)[0],
    "mae": mean_absolute_error,
    "mse": mean_squared_error,
}


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Spearman/Pearson/MAE/MSE on raw values - no cross-model rescaling."""
    return {name: float(fn(y_true, y_pred)) for name, fn in METRIC_FUNCS.items()}


def _metric_fn(metric: str) -> MetricFn:
    if metric not in METRIC_FUNCS:
        raise ValueError(
            f"Unknown metric: {metric!r}. Choose from {sorted(METRIC_FUNCS)}."
        )
    return METRIC_FUNCS[metric]


def _interval(deltas: np.ndarray, ci: float) -> tuple[float, float]:
    alpha = 1 - ci
    lower, upper = np.quantile(deltas, [alpha / 2, 1 - alpha / 2])
    return float(lower), float(upper)


def paired_bootstrap_delta_ci(
    y_true: np.ndarray,
    preds_a: np.ndarray,
    preds_b: np.ndarray,
    metric: str,
    n_resamples: int = 1000,
    ci: float = 0.95,
    seed: int = 42,
) -> tuple[float, float]:
    """CI for `metric(preds_a) - metric(preds_b)` under paired resampling.

    `metric` must be a key of `METRIC_FUNCS`. Returns `(lower, upper)` for the
    `ci` central interval (default 95%) over `n_resamples` bootstrap draws.

    Each draw resamples **rows**, which assumes the rows are independent. Where
    several rows are replicate measurements of one unit - the same sequence
    scored more than once - use `clustered_paired_bootstrap_delta_ci` instead,
    or this interval will be narrower than the data supports.
    """
    metric_fn = _metric_fn(metric)
    y_true = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)
    n = len(y_true)

    rng = np.random.default_rng(seed)
    deltas = np.empty(n_resamples)
    for i in range(n_resamples):
        idx = rng.integers(0, n, size=n)
        deltas[i] = metric_fn(y_true[idx], preds_a[idx]) - metric_fn(
            y_true[idx], preds_b[idx]
        )

    return _interval(deltas, ci)


def clustered_paired_bootstrap_delta_ci(
    y_true: np.ndarray,
    preds_a: np.ndarray,
    preds_b: np.ndarray,
    clusters: np.ndarray,
    metric: str,
    n_resamples: int = 1000,
    ci: float = 0.95,
    seed: int = 42,
) -> tuple[float, float]:
    """`paired_bootstrap_delta_ci` resampling whole clusters, not rows.

    `clusters` labels each row with the unit that is actually independent -
    for a sequence model scored on replicate rows, the sequence. Each draw
    takes as many cluster labels as there are, with replacement, and keeps
    every row of a drawn cluster, so a unit measured several times is drawn
    once rather than counted as several independent observations. Drawn
    samples therefore vary in row count, which is what a cluster bootstrap
    does: the number of independent units is what is held fixed.

    With one row per cluster this is exactly the row bootstrap.
    """
    metric_fn = _metric_fn(metric)
    y_true = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)
    clusters = np.asarray(clusters)
    if len(clusters) != len(y_true):
        raise ValueError(
            f"got {len(clusters)} cluster labels for {len(y_true)} rows; "
            "every row needs one"
        )

    _, inverse = np.unique(clusters, return_inverse=True)
    rows_of = [np.flatnonzero(inverse == c) for c in range(inverse.max() + 1)]
    n_clusters = len(rows_of)

    rng = np.random.default_rng(seed)
    deltas = np.empty(n_resamples)
    for i in range(n_resamples):
        drawn = rng.integers(0, n_clusters, size=n_clusters)
        idx = np.concatenate([rows_of[c] for c in drawn])
        deltas[i] = metric_fn(y_true[idx], preds_a[idx]) - metric_fn(
            y_true[idx], preds_b[idx]
        )

    return _interval(deltas, ci)


In [ ]:
%%writefile ablation.py
"""Model B physical-feature ablation + paired bootstrap (Step 4-1, issue #13).

Trains three XGBoost regressors - Embedding-only (8,960d), Physical-only
(4d), and Full (8,964d) - on the same dev split, seed, and hyperparameters
as production Model B (`nt_feature_stacking.py`'s Phase 4), then paired-
bootstraps Full against each ablated variant on the Testset. No cross-model
or cross-variant score rescaling is applied anywhere in this module.
"""

import numpy as np
import xgboost as xgb
from paired_bootstrap import compute_metrics, paired_bootstrap_delta_ci

# Same protocol as nt_feature_stacking.py's Phase 4 XGBoost fit - shared here
# so all three ablation variants (and production Model B) are provably using
# identical hyperparameters, not three independently-typed copies of them.
XGB_HYPERPARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "tree_method": "hist",
    "early_stopping_rounds": 20,
}

EMBEDDING_DIM = 8960
PHYSICAL_DIM = 4

# Bootstrap deltas are always Full minus the named ablated variant, so a
# negative delta_mse / positive delta_spearman means Full is better.
BOOTSTRAP_METRICS = ("spearman", "mse")


def split_ablation_features(
    X_full: np.ndarray, embedding_dim: int = EMBEDDING_DIM
) -> dict[str, np.ndarray]:
    """Slice a `(n, embedding_dim + physical_dim)` feature matrix into the 3 variants' inputs."""
    return {
        "embedding": X_full[:, :embedding_dim],
        "physical": X_full[:, embedding_dim:],
        "full": X_full,
    }


def train_ablation_variant(
    X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray
) -> xgb.XGBRegressor:
    """Train one XGBoost regressor under the shared `XGB_HYPERPARAMS` protocol."""
    model = xgb.XGBRegressor(**XGB_HYPERPARAMS)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model


def run_ablation(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    bootstrap_seed: int = 42,
    embedding_dim: int = EMBEDDING_DIM,
) -> dict:
    """Train all 3 variants, evaluate on the Testset, and paired-bootstrap Full vs each.

    Returns a dict with `ablation_results` (per-variant Spearman/Pearson/MAE/MSE
    on the Testset, raw/unrescaled) and `paired_bootstrap_95ci` (95% CI for
    `metric(full) - metric(variant)`, for each of `BOOTSTRAP_METRICS`).
    `embedding_dim` only needs overriding in tests against synthetic,
    lower-dimensional feature matrices.
    """
    train_splits = split_ablation_features(X_train, embedding_dim)
    val_splits = split_ablation_features(X_val, embedding_dim)
    test_splits = split_ablation_features(X_test, embedding_dim)

    # These three key names ("embedding_only"/"physical_only"/"full_model") are
    # mandated verbatim by the Standard Export Schema (plan_realize.md section
    # 4, `ablation_results`), which issue #17 must reproduce exactly. This is
    # a deliberate exception to CONTEXT.md's glossary guidance to avoid
    # "embedding_only" (there, it warns against conflating it with "NT-only
    # regression head performance", a different, unrelated pipeline) - kept
    # here only because it's always nested under `ablation_results`, which
    # disambiguates it as the Step 4(1) ablation variant, never that other
    # metric.
    result_key = {
        "embedding": "embedding_only",
        "physical": "physical_only",
        "full": "full_model",
    }

    metrics_by_variant = {}
    predictions_by_variant = {}
    for name in ("embedding", "physical", "full"):
        model = train_ablation_variant(
            train_splits[name], y_train, val_splits[name], y_val
        )
        preds = model.predict(test_splits[name])
        predictions_by_variant[name] = preds
        metrics_by_variant[name] = compute_metrics(y_test, preds)

    bootstrap_cis = {}
    for name in ("embedding", "physical"):
        for metric in BOOTSTRAP_METRICS:
            lower, upper = paired_bootstrap_delta_ci(
                y_test,
                predictions_by_variant["full"],
                predictions_by_variant[name],
                metric=metric,
                seed=bootstrap_seed,
            )
            bootstrap_cis[f"delta_{metric}_full_vs_{name}_only"] = [lower, upper]

    return {
        "ablation_results": {
            result_key[name]: metrics_by_variant[name]
            for name in ("embedding", "physical", "full")
        },
        "paired_bootstrap_95ci": bootstrap_cis,
    }


In [ ]:
%%writefile grouped_model_b.py
"""Leak-free Model B on the sequence-grouped split (issue #37).

Building blocks for `colab_grouped_model_b.ipynb`, which retrains Model B on
`grouped_split/` (#34/#35) instead of the stale, leaky `train.csv`/`val.csv`:

- Rows of `SaCas9_v4.xlsx` are joined to `grouped_split/assignments.csv` on
  `original_id`; train/val/test must share no sequence, and the test rows
  must match `grouped_split/test_metadata.csv` and its checksum.
- A new NT regression checkpoint is trained with the `nt_pretrain.py` recipe
  unchanged (`FINETUNE_ARGS` mirrors its `TrainingArguments`); only the data
  differs.
- Two XGBoost heads are trained with `ablation.XGB_HYPERPARAMS` on the
  `Model_B_Predictor` features: **all-rows** (one sample per replicate row,
  as in production) and **seq-mean** (one sample per sequence, labels
  averaged across its replicates).

The notebook needs a GPU for the fine-tune and feature extraction; everything
here is also runnable on CPU against small inputs, which is how it is tested.
"""

import importlib.metadata
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from ablation import XGB_HYPERPARAMS, train_ablation_variant
from grouped_split import SPLITS, normalize_sequences
from grouped_testset import column_checksum
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

CHECKPOINT_DIRNAME = "NT_sacas9_grouped_model"

SEED = 42
NT_BASE_MODEL = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
MAX_LENGTH = 64
EARLY_STOPPING_PATIENCE = 3
# nt_pretrain.py's TrainingArguments, minus its Colab-specific paths, plus an
# explicit seed (nt_pretrain.py relied on the Trainer default, also 42).
FINETUNE_ARGS = {
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 10,
    "weight_decay": 0.01,
    "fp16": True,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "logging_steps": 10,
    "seed": SEED,
}
# Disk only, not part of the recipe: each epoch checkpoint (weights + optimizer
# state) is ~6 GB, and the Trainer never deletes the best one.
CHECKPOINT_STORAGE_ARGS = {"save_total_limit": 2}

PREDICTION_COLUMNS = [
    "sample_id",
    "original_id",
    "sequence",
    "cell_line_block",
    "true_score",
    "pred_all_rows",
    "pred_seq_mean",
]
LIBRARIES = (
    "torch",
    "transformers",
    "accelerate",
    "xgboost",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "biopython",
)


def join_grouped_splits(
    full_df: pd.DataFrame, assignments: pd.DataFrame
) -> dict[str, pd.DataFrame]:
    """Dataset rows per split, joined on `original_id`, in source row order."""
    rows = full_df[["original_id", "sequence", "true_score"]].merge(
        assignments[["original_id", "normalized_sequence", "cell_line_block", "split"]],
        on="original_id",
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    unmatched = rows[rows["_merge"] != "both"]
    if len(unmatched):
        raise ValueError(
            f"{len(unmatched)} original_id values are in only one of the dataset "
            f"and assignments.csv, e.g. {unmatched['original_id'].head().tolist()}"
        )
    rows = rows.drop(columns="_merge").sort_values("original_id", kind="stable")
    return {
        split: rows[rows["split"] == split].drop(columns="split").reset_index(drop=True)
        for split in SPLITS
    }


def verify_no_sequence_overlap(splits: dict[str, pd.DataFrame]) -> None:
    """No normalized sequence may appear in more than one split."""
    for a, b in combinations(SPLITS, 2):
        shared = set(normalize_sequences(splits[a]["sequence"])) & set(
            normalize_sequences(splits[b]["sequence"])
        )
        if shared:
            raise ValueError(
                f"{a}/{b} sequence overlap: {len(shared)} sequences, "
                f"e.g. {sorted(shared)[:3]}"
            )


def verify_grouped_testset(
    test_df: pd.DataFrame, testset_metadata: pd.DataFrame, checksum_record: dict
) -> None:
    """Joined test rows must equal `grouped_split/test_metadata.csv` + its checksum."""
    if (
        column_checksum(testset_metadata["sequence"])
        != checksum_record["dataset_sha256"]
    ):
        raise ValueError(
            "grouped_split/test_metadata.csv does not match the dataset checksum "
            "in grouped_split/test_checksum.json"
        )
    if len(test_df) != checksum_record["rows"]:
        raise ValueError(
            f"{len(test_df)} test rows, expected {checksum_record['rows']}"
        )

    columns = ["original_id", "sequence", "true_score", "cell_line_block"]
    try:
        pd.testing.assert_frame_equal(
            test_df[columns].reset_index(drop=True),
            testset_metadata[columns].reset_index(drop=True),
            check_dtype=False,
        )
    except AssertionError as err:
        raise ValueError(
            f"Grouped test rows do not match grouped_split/test_metadata.csv: {err}"
        ) from err


def finetune_compute_metrics(eval_pred) -> dict[str, float]:
    """`nt_pretrain.py`'s `compute_metrics`, unchanged."""
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    if predictions.ndim > 1:
        predictions = predictions.flatten()
    if labels.ndim > 1:
        labels = labels.flatten()

    if np.std(predictions) == 0 or np.std(labels) == 0:
        return {"spearman_rho": 0.0, "pearson_r": 0.0, "mae": 0.0, "mse": 0.0}

    return {
        "spearman_rho": float(spearmanr(labels, predictions)[0]),
        "pearson_r": float(pearsonr(labels, predictions)[0]),
        "mae": float(mean_absolute_error(labels, predictions)),
        "mse": float(mean_squared_error(labels, predictions)),
    }


def _tokenize(tokenizer, df: pd.DataFrame) -> list[dict]:
    """`nt_pretrain.py`'s `preprocess_function`, as a list the Trainer can index."""
    tokenized = tokenizer(
        df["sequence"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    return [
        {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": float(label),
        }
        for input_ids, attention_mask, label in zip(
            tokenized["input_ids"],
            tokenized["attention_mask"],
            df["true_score"],
            strict=True,
        )
    ]


def finetune_nt(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    output_dir: str | Path,
    model_name: str = NT_BASE_MODEL,
    seed: int = SEED,
    overrides: dict | None = None,
):
    """Fine-tune NT with the `nt_pretrain.py` recipe; returns `(trainer, tokenizer)`.

    The trainer holds the best (lowest val loss) checkpoint after training.
    `overrides` exists only so tests can run the recipe on CPU.
    """
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        EarlyStoppingCallback,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=1, ignore_mismatched_sizes=True
    )
    args = TrainingArguments(
        output_dir=str(output_dir),
        **{
            **FINETUNE_ARGS,
            **CHECKPOINT_STORAGE_ARGS,
            "seed": seed,
            **(overrides or {}),
        },
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=_tokenize(tokenizer, train_df),
        eval_dataset=_tokenize(tokenizer, val_df),
        compute_metrics=finetune_compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)
        ],
    )
    trainer.train()
    return trainer, tokenizer


def summarize_finetune_history(log_history: list[dict]) -> dict:
    """Per-epoch val metrics from `trainer.state.log_history`, plus the best epoch."""
    epochs = [
        {k: v for k, v in entry.items() if k == "epoch" or k.startswith("eval_")}
        for entry in log_history
        if "eval_loss" in entry
    ]
    best = min(epochs, key=lambda e: e["eval_loss"])
    return {
        "epochs": epochs,
        "best_epoch": best["epoch"],
        "best_eval_loss": best["eval_loss"],
    }


def sequence_mean(
    X: np.ndarray, y: np.ndarray, groups: pd.Series
) -> tuple[np.ndarray, np.ndarray]:
    """One sample per sequence (first-occurrence order), labels averaged per sequence.

    Replicates of a sequence must carry identical features, since features
    are a function of the sequence alone.
    """
    codes, _ = pd.factorize(pd.Series(groups).reset_index(drop=True))
    first_row = pd.Series(np.arange(len(codes))).groupby(codes).first().to_numpy()
    if not np.array_equal(X, X[first_row[codes]]):
        raise ValueError("Replicates of the same sequence have different features")

    counts = np.bincount(codes)
    return X[first_row], np.bincount(codes, weights=y) / counts


@dataclass
class HeadFit:
    model: xgb.XGBRegressor
    n_train_samples: int
    n_val_samples: int


def train_heads(
    X_train: np.ndarray,
    y_train: np.ndarray,
    groups_train: pd.Series,
    X_val: np.ndarray,
    y_val: np.ndarray,
    groups_val: pd.Series,
) -> dict[str, HeadFit]:
    """The all-rows and seq-mean XGBoost heads, both early-stopped on val."""
    X_train_seq, y_train_seq = sequence_mean(X_train, y_train, groups_train)
    X_val_seq, y_val_seq = sequence_mean(X_val, y_val, groups_val)
    datasets = {
        "all_rows": (X_train, y_train, X_val, y_val),
        "seq_mean": (X_train_seq, y_train_seq, X_val_seq, y_val_seq),
    }
    return {
        name: HeadFit(
            model=train_ablation_variant(X_fit, y_fit, X_eval, y_eval),
            n_train_samples=len(X_fit),
            n_val_samples=len(X_eval),
        )
        for name, (X_fit, y_fit, X_eval, y_eval) in datasets.items()
    }


def build_test_predictions(
    testset_metadata: pd.DataFrame,
    pred_all_rows: np.ndarray,
    pred_seq_mean: np.ndarray,
) -> pd.DataFrame:
    """`test_predictions.csv`: one row per grouped Testset row."""
    n = len(testset_metadata)
    if len(pred_all_rows) != n or len(pred_seq_mean) != n:
        raise ValueError(
            f"Expected {n} predictions per head, got "
            f"{len(pred_all_rows)} (all-rows) and {len(pred_seq_mean)} (seq-mean)"
        )
    table = testset_metadata.reset_index(drop=True).copy()
    table["pred_all_rows"] = np.asarray(pred_all_rows, dtype=float)
    table["pred_seq_mean"] = np.asarray(pred_seq_mean, dtype=float)
    return table[PREDICTION_COLUMNS]


def library_versions() -> dict[str, str | None]:
    """Installed version of each of `LIBRARIES`, `None` if missing."""
    versions = {}
    for name in LIBRARIES:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def build_training_log(
    log_history: list[dict],
    heads: dict[str, HeadFit],
    n_rows: dict[str, int],
    seed: int = SEED,
) -> dict:
    """`training_log.json`: fine-tune val metrics, XGBoost fits, seed, versions."""
    return {
        "seed": seed,
        "n_rows": n_rows,
        "recipe": {
            "base_model": NT_BASE_MODEL,
            "max_length": MAX_LENGTH,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "finetune_args": FINETUNE_ARGS,
            "xgb_hyperparams": XGB_HYPERPARAMS,
        },
        "finetune": summarize_finetune_history(log_history),
        "xgboost": {
            name: {
                "best_iteration": int(fit.model.best_iteration),
                "best_score": float(fit.model.best_score),
                "n_train_samples": fit.n_train_samples,
                "n_val_samples": fit.n_val_samples,
            }
            for name, fit in heads.items()
        },
        "library_versions": library_versions(),
    }


## 5. Confirm GPU is available

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Switch the runtime to GPU (Runtime > Change runtime type)."
)
print(torch.cuda.get_device_name(0))

## 6. Load the grouped split and verify it

Joins `SaCas9_v4.xlsx` to `grouped_split/assignments.csv` on `original_id`,
then refuses to continue unless train/val/test share no sequence and the test
rows are exactly `grouped_split/test_metadata.csv` (checksum from
`grouped_split/test_checksum.json`).

In [ ]:
import json

import pandas as pd
from grouped_model_b import (
    join_grouped_splits,
    verify_grouped_testset,
    verify_no_sequence_overlap,
)
from grouped_split import load_full_dataset

full_df = load_full_dataset("SaCas9_v4.xlsx")
assignments = pd.read_csv("grouped_split/assignments.csv")
splits = join_grouped_splits(full_df, assignments)
verify_no_sequence_overlap(splits)

testset_metadata = pd.read_csv("grouped_split/test_metadata.csv")
with open("grouped_split/test_checksum.json", encoding="utf-8") as f:
    checksum_record = json.load(f)
verify_grouped_testset(splits["test"], testset_metadata, checksum_record)

train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]
n_rows = {name: len(df) for name, df in splits.items()}
assert n_rows == {"train": 4118, "val": 513, "test": 514}, n_rows
print("Rows:", n_rows)
n_sequences = {name: df["normalized_sequence"].nunique() for name, df in splits.items()}
print("Sequences:", n_sequences)
print("Zero sequence overlap; Testset matches grouped_split/test_metadata.csv.")

## 7. Fine-tune NT (`nt_pretrain.py` recipe, grouped data)

Up to 10 epochs with early stopping (patience 3) on val loss; the best
checkpoint is reloaded at the end and saved to `DRIVE_DIR/NT_sacas9_grouped_model/`
with its tokenizer. Per-epoch checkpoints go to local Colab disk and are deleted
afterwards. The Trainer's `log_history` is also saved to Drive right away.

If the runtime disconnects after this step, rerun steps 1-6 with
`REUSE_SAVED_CHECKPOINT = True` to reuse the saved checkpoint and log instead
of fine-tuning again. Leave it `False` for a fresh run.

In [ ]:
import shutil
import time

from grouped_model_b import (
    CHECKPOINT_DIRNAME,
    SEED,
    finetune_nt,
    summarize_finetune_history,
)

REUSE_SAVED_CHECKPOINT = False
LOCAL_CHECKPOINTS = "/content/nt_grouped_checkpoints"
checkpoint_dir = os.path.join(DRIVE_DIR, CHECKPOINT_DIRNAME)
log_history_path = os.path.join(DRIVE_DIR, "grouped_finetune_log_history.json")

if REUSE_SAVED_CHECKPOINT:
    for path in (log_history_path, os.path.join(checkpoint_dir, "config.json")):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Nothing to reuse: {path} is missing")
    print(f"Reusing {checkpoint_dir} and {log_history_path}")
    with open(log_history_path, encoding="utf-8") as f:
        finetune_log_history = json.load(f)
else:
    t0 = time.time()
    trainer, tokenizer = finetune_nt(train_df, val_df, LOCAL_CHECKPOINTS, seed=SEED)
    print(f"Fine-tune completed in {(time.time() - t0) / 60:.1f} min")

    trainer.save_model(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)
    finetune_log_history = trainer.state.log_history
    with open(log_history_path, "w", encoding="utf-8") as f:
        json.dump(finetune_log_history, f, indent=2)
    print(f"Best checkpoint saved to {checkpoint_dir}")

    del trainer
    torch.cuda.empty_cache()
    shutil.rmtree(LOCAL_CHECKPOINTS, ignore_errors=True)

finetune_summary = summarize_finetune_history(finetune_log_history)
for epoch in finetune_summary["epochs"]:
    print(epoch)
print(
    f"Best epoch {finetune_summary['best_epoch']}, "
    f"val loss {finetune_summary['best_eval_loss']:.4f}"
)

## 8. Extract features with `Model_B_Predictor`

Loads the saved checkpoint exactly as the local `Model_B_Predictor` will, and
builds features with its own `extract_nt_embeddings` + `compute_physical_features`
(batched with `ism_sweep.dispatch_in_batches`, as in `colab_ablation.ipynb`).
Features depend only on the sequence, so each distinct sequence is embedded
once and shared by its replicate rows. Lower `PREDICT_BATCH_SIZE` on CUDA OOM.

In [ ]:
import numpy as np
from ism_sweep import dispatch_in_batches
from model_b_wrapper import Model_B_Predictor

predictor = Model_B_Predictor(
    nt_model_dir=checkpoint_dir, xgb_model_path="hybrid_xgb_model.json"
)

PREDICT_BATCH_SIZE = 128  # lower (e.g. 32) if you hit CUDA OOM


def build_features(model_b, sequences):
    embeddings = dispatch_in_batches(
        model_b.extract_nt_embeddings, sequences, PREDICT_BATCH_SIZE, stack=np.vstack
    )
    physical = model_b.compute_physical_features(sequences)
    return np.hstack([embeddings, physical])


unique_sequences = sorted(
    set(train_df["sequence"])
    | set(val_df["sequence"])
    | set(testset_metadata["sequence"])
)
t0 = time.time()
X_unique = build_features(predictor, unique_sequences)
print(f"Feature extraction completed in {(time.time() - t0) / 60:.1f} min")
row_of = {seq: i for i, seq in enumerate(unique_sequences)}


def features_for(df):
    return X_unique[[row_of[seq] for seq in df["sequence"]]]


X_train, y_train = features_for(train_df), train_df["true_score"].to_numpy()
X_val, y_val = features_for(val_df), val_df["true_score"].to_numpy()
X_test = features_for(testset_metadata)
assert X_unique.shape == (len(unique_sequences), 8964), X_unique.shape
print(f"X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

del predictor
torch.cuda.empty_cache()

## 9. Train the all-rows and seq-mean XGBoost heads

In [ ]:
from grouped_model_b import train_heads
from paired_bootstrap import compute_metrics

heads = train_heads(
    X_train,
    y_train,
    train_df["normalized_sequence"],
    X_val,
    y_val,
    val_df["normalized_sequence"],
)
pred_all_rows = heads["all_rows"].model.predict(X_test)
pred_seq_mean = heads["seq_mean"].model.predict(X_test)

y_test = testset_metadata["true_score"].to_numpy()
for name, fit in heads.items():
    print(
        f"{name}: {fit.n_train_samples} train samples, "
        f"best_iteration={fit.model.best_iteration}"
    )
# Quick look only; the full evaluation is a separate step (issue #38).
print("Row-level Testset metrics, all-rows:", compute_metrics(y_test, pred_all_rows))
print("Row-level Testset metrics, seq-mean:", compute_metrics(y_test, pred_seq_mean))

## 10. Save `grouped_model_b/`

Written to `DRIVE_DIR/grouped_model_b/` and downloaded as a zip.

In [ ]:
from grouped_model_b import build_test_predictions, build_training_log

os.makedirs("grouped_model_b", exist_ok=True)
heads["all_rows"].model.save_model("grouped_model_b/xgb_all_rows.json")
heads["seq_mean"].model.save_model("grouped_model_b/xgb_seq_mean.json")

predictions = build_test_predictions(testset_metadata, pred_all_rows, pred_seq_mean)
assert len(predictions) == 514
predictions.to_csv(
    "grouped_model_b/test_predictions.csv", index=False, lineterminator="\n"
)

training_log = build_training_log(finetune_log_history, heads, seed=SEED, n_rows=n_rows)
with open("grouped_model_b/training_log.json", "w", encoding="utf-8") as f:
    json.dump(training_log, f, indent=2)

print(sorted(os.listdir("grouped_model_b")))

## 11. Check the checkpoint loads in `Model_B_Predictor`

Loads the Drive checkpoint with the all-rows head the way the local code will,
and predicts the Testset end to end. Embeddings are batched differently from
step 8, so GPU float noise can move a few predictions slightly (an XGBoost split
can tip); the check is on rank agreement, with the max difference printed.

In [ ]:
from ism_sweep import predict_in_batches

reloaded = Model_B_Predictor(
    nt_model_dir=checkpoint_dir, xgb_model_path="grouped_model_b/xgb_all_rows.json"
)
reloaded_preds = predict_in_batches(
    reloaded, testset_metadata["sequence"].tolist(), PREDICT_BATCH_SIZE
)
from scipy.stats import spearmanr

max_diff = float(np.max(np.abs(reloaded_preds - pred_all_rows)))
agreement = spearmanr(reloaded_preds, pred_all_rows)[0]
print(f"Max |reloaded - pred_all_rows| = {max_diff:.2e}, Spearman {agreement:.6f}")
assert agreement > 0.999, "Reloaded Model_B_Predictor disagrees with pred_all_rows"

## 12. Download the results

In [ ]:
from google.colab import files

shutil.make_archive("/content/grouped_model_b", "zip", DRIVE_DIR, "grouped_model_b")
files.download("/content/grouped_model_b.zip")

Unzip into your local `xai_test/` (giving `xai_test/grouped_model_b/`) and commit
it. Download `NT_sacas9_grouped_model/` from Drive into `xai_test/` separately;
don't commit it.